## Visium HD for ClusterDE spatial marker gene analysis

This notebook can be used to read in the spatial transcriptomics colorectal cancer dataset from [Oliveira et al., Nature Genetics 2025](https://doi.org/10.1038/s41588-025-02193-3). At the end, it creates and saves an AnnData object storing the experimental results. The counts are from [GSE280315](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE280315) and the spatial annotations are from [HumanColonCancer_VisiumHD](https://github.com/10XGenomics/HumanColonCancer_VisiumHD). We deliberately don't normalize any data. You can switch to different sections from this paper by changing `SAMPLE` to either `"P1CRC"` or `"P5CRC"`.

In [ ]:
from pathlib import Path
import anndata as ad
from processing.sources import download
from processing.visium import load_section

SAMPLE = "P2CRC"
raw = Path("data/raw")
processed = Path("data/processed")

We download the selected section's count matrix, tissue positions, scale factors, and annotations, then load them. Positions and annotations are joined to the count matrix by barcode, so the matrix defines which bins are kept.

In [ ]:
paths = download(raw, SAMPLE)
adata = load_section(paths, SAMPLE)

print(f"""Bins x genes: {adata.shape}
Bins with no counts: {int((adata.obs["total_counts"] == 0).sum())}
Genes with no counts: {int((adata.X.getnnz(axis=0) == 0).sum())}
Singlet bins: {int((adata.obs["DeconvolutionClass"] == "singlet").sum())}

{adata.obs["DeconvolutionClass"].value_counts(dropna=False)}

Fraction of bins with a label:
{adata.obs.select_dtypes("category").notna().mean()}""")

Bins x genes: (545913, 18085)
Bins with no counts: 63
Genes with no counts: 13
Singlet bins: 322327

DeconvolutionClass
singlet              322327
NaN                  114909
doublet_certain       45436
doublet_uncertain     37514
reject                25727
Name: count, dtype: int64

Fraction of bins with a label:
sample_id              1.000000
DeconvolutionClass     0.789510
DeconvolutionLabel1    0.789510
DeconvolutionLabel2    0.789510
Periphery              1.000000
UnsupervisedL1         1.000000
UnsupervisedL2         1.000000
MacrophageSubtype      0.003284
GobletSubcluster       0.048772
cell_type              0.590437
dtype: float64


We save the object with gzip compression, release the in-memory copy, and reopen the file in backed mode to confirm it is readable.

In [3]:
processed.mkdir(parents=True, exist_ok=True)
output = processed / f"{SAMPLE}_008um.h5ad"
adata.write_h5ad(output, compression="gzip")
del adata

saved = ad.read_h5ad(output, backed="r")
print("Saved", output, saved.shape)
saved.file.close()

Saved data/processed/P2CRC_008um.h5ad (545913, 18085)
